#  Task parallelism
* Task is a more general mechanism to extract parallelism and distribute computation (called a task) _dynamically_ to threads in a team created by `#pragma omp parallel`
* A thread can create a task at any point in the execution of a parallel region and they are dispatched to available threads at runtime
* As a thread can create a task at any point, a task can create another task. that is, parallelism can be arbitrarily nested and the number of tasks can be difficult to predict (unlike the number of iterations of a for loop)
* A common pattern
  1. enter a parallel region by `#pragma omp parallel`
  1. ensure the statement is executed by only a single (root) thread with [#pragma omp master](https://www.openmp.org/spec-html/5.2/openmpse24.html#x118-4380002.16)
  1. create tasks at any point by [#pragma omp task](https://www.openmp.org/spec-html/5.2/openmpsu46.html#x70-2000002.10.1)
  1. a task waits for tasks it created to finish by [#pragma omp taskwait](https://www.openmp.org/spec-html/5.2/openmpsu93.html#x124-4690002.17.5)

* Let's see the effect of `#pragma omp master` first (without creating any task)


In [ ]:
%%writefile omp_master.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  double t0 = omp_get_wtime();
#pragma omp parallel
  {
#pragma omp master
    printf("inside the master pragma: I am thread %d of a team of %d threads\n",
           omp_get_thread_num(), omp_get_num_threads());
    printf("out of the master pragma: I am thread %d of a team of %d threads\n",
           omp_get_thread_num(), omp_get_num_threads());
  }
  double t1 = omp_get_wtime();
  printf("%f sec\n", t1 - t0);
  return 0;
}

In [ ]:
%%bash
clang -fopenmp omp_master.c -o omp_master
# nvc -mp omp_master.c -o omp_master

In [ ]:
%%bash
OMP_NUM_THREADS=3 ./omp_master

* Since this is a common idiom, they can be combined into one pragma (`#pragma omp parallel master`)
  * <font color=red>This feature is not supported by NVIDIA compiler, however</font>
* The program below creates a parallel region whose entire region is executed only by the master and thus does not serve any useful purpose but mere a demonstration of the feature


In [ ]:
%%writefile omp_parallel_master.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  double t0 = omp_get_wtime();
#pragma omp parallel master
  printf("I am thread %d of a team of %d threads\n",
         omp_get_thread_num(), omp_get_num_threads());
  double t1 = omp_get_wtime();
  printf("%f sec\n", t1 - t0);
  return 0;
}

In [ ]:
%%bash
clang -fopenmp omp_parallel_master.c -o omp_parallel_master
# NVIDIA compiler does not support this program
# nvc -mp omp_parallel_master.c -o omp_parallel_master

In [ ]:
%%bash
OMP_NUM_THREADS=3 ./omp_parallel_master

* Let's create a few tasks now


In [ ]:
%%writefile omp_task.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  double t0 = omp_get_wtime();
#pragma omp parallel
#pragma omp master
  {
    printf("I am thread %d of a team of %d threads\n",
           omp_get_thread_num(), omp_get_num_threads());
#pragma omp task
    {
      printf("task A executed by %d of %d\n", omp_get_thread_num(), omp_get_num_threads());
      usleep(500 * 1000);
    }
#pragma omp task
    {
      printf("task B executed by %d of %d\n", omp_get_thread_num(), omp_get_num_threads());
      usleep(1000 * 1000);
    }
#pragma omp taskwait
    printf("two tasks done, executed by %d of %d\n", omp_get_thread_num(), omp_get_num_threads());
  }
  double t1 = omp_get_wtime();
  printf("%f sec\n", t1 - t0);
  return 0;
}

In [ ]:
%%bash
clang -fopenmp omp_task.c -o omp_task
# nvc -mp omp_task.c -o omp_task

In [ ]:
%%bash
OMP_NUM_THREADS=3 ./omp_task

* Tasks are particularly good at parallel recursions, as the following program demonstrates
* This is a common pattern that appears in many algorithms, particularly divide-and-conquer algorithms


In [ ]:
%%writefile omp_rec_task.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

void recursive_tasks(int n, int tid) {
  printf("task %d by %d of %d\n",
         tid, omp_get_thread_num(), omp_get_num_threads());
  fflush(stdout);
  if (n == 0) {
    usleep(300 * 1000);
  } else {
#pragma omp task
    recursive_tasks(n - 1, 2 * tid + 1);
#pragma omp task
    recursive_tasks(n - 1, 2 * tid + 2);
#pragma omp taskwait
  }
}
int main() {
  double t0 = omp_get_wtime();
#pragma omp parallel
#pragma omp master
  {
    recursive_tasks(5, 0);
  }
  double t1 = omp_get_wtime();
  printf("%f sec\n", t1 - t0);
  return 0;
}

In [ ]:
%%bash
clang -fopenmp omp_rec_task.c -o omp_rec_task
# nvc -mp omp_rec_task.c -o omp_rec_task

In [ ]:
%%bash
OMP_NUM_THREADS=10 ./omp_rec_task

# <font color="green"> Problem 1 :  A quiz about recursive tasks</font>
* Answer the following questions
* How many tasks are created by `recursive_tasks(n, 0)`?  Include the caller of `recursive_tasks(n, 0)` as a task.  i.e., consider `recursive_tasks(0, 0)` creates one task
* How many of them are leaf tasks?
* Express them in terms of $n$


* Approximately what is the ideal execution time of `recursive_tasks(5, 0)` when using 10 threads?
* Compare it with what you observed


* Name an algorithm or two for which recursive tasks will be useful for parallelizing it and explain why you think so


# 1. Taskloop
* As you can easily imagine, tasks can handle general nested loops if they can handle recursions
* Recent OpenMP actually has a construct just for that, which is [#pragma omp taskloop](https://www.openmp.org/spec-html/5.2/openmpsu47.html#x71-2080002.10.2)
  * <font color=red>This feature is not supported by NVIDIA compiler</font>
* Here is a demonstration showing it can handle non perfectly-nested loops


In [ ]:
%%writefile omp_taskloop.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  double t0 = omp_get_wtime();
#pragma omp parallel
#pragma omp master
#pragma omp taskloop
  for (int i = 0; i < 5; i++) {
    printf("i = %d starts\n", i);
    fflush(stdout);
#pragma omp taskloop
    for (int j = 0; j < 5; j++) {
      usleep(100 * 1000 * (i + j));
      printf("iteration (%d, %d) executed by thread %d\n", i, j, omp_get_thread_num());
      fflush(stdout);
    }
  }
  double t1 = omp_get_wtime();
  printf("%f sec\n", t1 - t0);
  return 0;
}

In [ ]:
%%bash
clang -fopenmp omp_taskloop.c -o omp_taskloop
# nvc -mp omp_taskloop.c -o omp_taskloop

In [ ]:
%%bash
OMP_NUM_THREADS=3 ./omp_taskloop

<font color="red">NOTE:</font>

* Implementing task requires a more general mechanism than work-sharing for statement; the former should be able to distribute tasks generated in the course of execution, whereas the former merely needs to distribute iterations that are easily identifiable at the point of entering `#pragma omp for`, thanks to the ["canonical form" restriction](https://www.openmp.org/spec-html/5.2/openmpsu40.html#x63-1260002.9.1)
* Task scheduling is always dynamic whereas work-sharing for (particularly with static scheduling) gives you more control and predictability about which thread executes which iteration
* This is a reason why two mechanisms which are seemingly redundant exist, besides a historical reason that initially there was not a tasking construct in OpenMP
